In [19]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt

from data_loader import build_complete_dataset
from window import create_record_windows

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix,
                             roc_curve)
from sklearn.model_selection import ParameterGrid

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import tensorflow.keras.backend as K


In [20]:

def extract_window_features(window_df, fs, prefix=""):
    features = {}
    columns = [column for column in window_df.columns if column != "Time"]
    data = window_df.drop(columns=["Time"], errors="ignore").values
    for i, col in enumerate(columns):
        signal = data[:, i]
        features[f"{prefix}_{col}_mean"] = np.mean(signal)
        features[f"{prefix}_{col}_std"] = np.std(signal)
        features[f"{prefix}_{col}_rms"] = np.sqrt(np.mean(signal ** 2))
        features[f"{prefix}_{col}_max"] = np.max(signal)
        features[f"{prefix}_{col}_min"] = np.min(signal)
        features[f"{prefix}_{col}_ptp"] = np.ptp(signal)
        features[f"{prefix}_{col}_skew"] = skew(signal) if len(signal) > 3 else 0
        features[f"{prefix}_{col}_kurtosis"] = kurtosis(signal) if len(signal) > 3 else 0
        fft_vals = np.abs(np.fft.rfft(signal))
        freqs = np.fft.rfftfreq(len(signal), d=1 / fs)
        features[f"{prefix}_{col}_fft_energy"] = np.sum(fft_vals ** 2)
        features[f"{prefix}_{col}_fft_peak_freq"] = freqs[np.argmax(fft_vals)]
    return features


In [21]:

def process_record_features(record, create_record_windows_func):
    acc_windows, gyro_windows, mic_windows = create_record_windows_func(record)

    record_features = []
    min_len = min(len(acc_windows), len(gyro_windows), len(mic_windows))

    for i in range(min_len):
        features = {}
        features.update(extract_window_features(acc_windows[i], fs=6700, prefix="acc"))
        features.update(extract_window_features(gyro_windows[i], fs=6700, prefix="gyro"))
        features.update(extract_window_features(mic_windows[i], fs=16000, prefix="mic"))

        features["segment_id"] = record.metadata.get("segment_id")
        features["split_label"] = record.metadata.get("split_label")
        features["anomaly_label"] = record.metadata.get("anomaly_label")
        features["domain_shift_op"] = record.metadata.get("domain_shift_op")
        features["domain_shift_env"] = record.metadata.get("domain_shift_env")
        record_features.append(features)

    return pd.DataFrame(record_features)


In [22]:

def build_feature_dataset(dataset, create_record_windows_func):
    results = []
    for record in tqdm(dataset, desc="Ekstrakcja cech"):
        df_feats = process_record_features(record, create_record_windows_func)
        results.append(df_feats)
    return pd.concat(results, ignore_index=True)

In [23]:


path = Path("../data")
train_path = path / "X_train.csv"
test_path = path / "X_test.csv"

if train_path.exists() and test_path.exists():
    X_train_df = pd.read_csv(train_path)
    X_test_df = pd.read_csv(test_path)
else:
    df = build_complete_dataset()
    X_train_df = build_feature_dataset(df[0], create_record_windows)
    X_test_df = build_feature_dataset(df[1], create_record_windows)
    X_train_df.to_csv(train_path, index=False)
    X_test_df.to_csv(test_path, index=False)

print("Train:", X_train_df.shape)
print("Test :", X_test_df.shape)

print(X_train_df.columns.tolist())
print(X_test_df.columns.tolist())

X_train_source = X_train_df[X_train_df["split_label"] == "Normal_Source_Train"].copy()
X_train_target = X_train_df[X_train_df["split_label"] == "Normal_Target_Train"].copy()

X_val = X_test_df[
    X_test_df["split_label"].isin(["Normal_Source_Test", "Anomaly_Source_Test"])].copy()  # zbior walidacyjny

X_test = X_test_df[
    X_test_df["split_label"].isin(["Normal_Target_Test", "Anomaly_Target_Test"])].copy()  # to nasz zbiór testowy
metadata_cols = ["segment_id", "anomaly_label", "domain_shift_op", "domain_shift_env"]

acc_features = [col for col in X_train_df.columns if col.startswith("acc_")]
gyro_features = [col for col in X_train_df.columns if col.startswith("gyro_")]
mic_features = [col for col in X_train_df.columns if col.startswith("mic_")]
X_train_acc = X_train_df[acc_features].fillna(0)
X_test_acc = X_test[acc_features].fillna(0)

X_train_mic = X_train_df[mic_features].fillna(0)
X_test_mic = X_test[mic_features].fillna(0)

X_train_gyro = X_train_df[gyro_features].fillna(0)
X_test_gyro = X_test[gyro_features].fillna(0)

X_train_source_acc = X_train_source[acc_features].fillna(0)
X_train_source_mic = X_train_source[mic_features].fillna(0)
X_train_source_gyro = X_train_source[gyro_features].fillna(0)

X_train_target_acc = X_train_target[acc_features].fillna(0)
X_train_target_mic = X_train_target[mic_features].fillna(0)
X_train_target_gyro = X_train_target[gyro_features].fillna(0)

X_val_acc = X_val[acc_features].fillna(0)
X_val_mic = X_val[mic_features].fillna(0)
X_val_gyro = X_val[gyro_features].fillna(0)

X_test_acc = X_test[acc_features].fillna(0)
X_test_mic = X_test[mic_features].fillna(0)
X_test_gyro = X_test[gyro_features].fillna(0)


Train: (102984, 75)
Test : (25984, 75)
['acc_A_x [g]_mean', 'acc_A_x [g]_std', 'acc_A_x [g]_rms', 'acc_A_x [g]_max', 'acc_A_x [g]_min', 'acc_A_x [g]_ptp', 'acc_A_x [g]_skew', 'acc_A_x [g]_kurtosis', 'acc_A_x [g]_fft_energy', 'acc_A_x [g]_fft_peak_freq', 'acc_A_y [g]_mean', 'acc_A_y [g]_std', 'acc_A_y [g]_rms', 'acc_A_y [g]_max', 'acc_A_y [g]_min', 'acc_A_y [g]_ptp', 'acc_A_y [g]_skew', 'acc_A_y [g]_kurtosis', 'acc_A_y [g]_fft_energy', 'acc_A_y [g]_fft_peak_freq', 'acc_A_z [g]_mean', 'acc_A_z [g]_std', 'acc_A_z [g]_rms', 'acc_A_z [g]_max', 'acc_A_z [g]_min', 'acc_A_z [g]_ptp', 'acc_A_z [g]_skew', 'acc_A_z [g]_kurtosis', 'acc_A_z [g]_fft_energy', 'acc_A_z [g]_fft_peak_freq', 'gyro_G_x [mdps]_mean', 'gyro_G_x [mdps]_std', 'gyro_G_x [mdps]_rms', 'gyro_G_x [mdps]_max', 'gyro_G_x [mdps]_min', 'gyro_G_x [mdps]_ptp', 'gyro_G_x [mdps]_skew', 'gyro_G_x [mdps]_kurtosis', 'gyro_G_x [mdps]_fft_energy', 'gyro_G_x [mdps]_fft_peak_freq', 'gyro_G_y [mdps]_mean', 'gyro_G_y [mdps]_std', 'gyro_G_y [mdps]_

In [24]:

def find_best_threshold(y_true, scores):
    scores = np.asarray(scores)
    y_true = np.asarray(y_true)
    thresholds = np.percentile(scores, np.linspace(0, 100, 200))
    best_threshold = thresholds[0]
    best_f1 = -1
    for threshold in thresholds:
        y_pred = (scores >= threshold).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    return best_threshold, best_f1


In [25]:

def build_autoencoder(input_dim, encoding_dim=16):
    input_layer = Input(shape=(input_dim,))

    encoded = Dense(64, activation='relu')(input_layer)
    encoded = Dense(32, activation='relu')(encoded)
    encoded = Dense(encoding_dim, activation='relu')(encoded)

    decoded = Dense(32, activation='relu')(encoded)
    decoded = Dense(64, activation='relu')(decoded)
    decoded = Dense(input_dim, activation='linear')(decoded)

    autoencoder = Model(inputs=input_layer, outputs=decoded)
    autoencoder.compile(optimizer='adam', loss='mse')

    return autoencoder


In [26]:

def tune_autoencoder(
        X_train,
        X_val,
        train_df,
        val_df,
        sensor_type,
        param_grid):
    results = []
    y_val = (val_df.groupby("segment_id")["anomaly_label"].first().eq("loosescrewsA").astype(int))

    for params in ParameterGrid(param_grid):
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        model = build_autoencoder(
            input_dim=X_train_scaled.shape[1],
            encoding_dim=params["encoding_dim"]
        )

        early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

        model.fit(
            X_train_scaled, X_train_scaled,
            epochs=params["epochs"],
            batch_size=params["batch_size"],
            validation_split=0.1,
            callbacks=[early_stop],
            verbose=0
        )

        X_val_pred = model.predict(X_val_scaled, verbose=0)
        val_scores = np.mean(np.power(X_val_scaled - X_val_pred, 2), axis=1)

        val_results = pd.DataFrame({"segment_id": val_df["segment_id"].values, "score": val_scores})
        segment_scores = (val_results.groupby("segment_id").agg(score=("score", "max")).reset_index())
        segment_scores = segment_scores.merge(y_val.rename("y_true"), left_on="segment_id", right_index=True,
                                              how="inner")

        threshold, best_f1 = find_best_threshold(segment_scores["y_true"].values, segment_scores["score"].values)
        y_pred = (segment_scores["score"] >= threshold).astype(int)

        precision = precision_score(segment_scores["y_true"], y_pred, zero_division=0)
        recall = recall_score(segment_scores["y_true"], y_pred, zero_division=0)

        accuracy = accuracy_score(segment_scores["y_true"], y_pred)
        roc_auc = roc_auc_score(segment_scores["y_true"], segment_scores["score"])

        results.append({"sensor": sensor_type,
                        **params,
                        "threshold": threshold,
                        "Accuracy": accuracy,
                        "Precision": precision,
                        "Recall": recall,
                        "ROC AUC": roc_auc,
                        "F1": best_f1})

    results_df = pd.DataFrame(results)

    best_row = results_df.loc[results_df["F1"].idxmax()]

    best_params = {
        "encoding_dim": int(best_row["encoding_dim"]),
        "epochs": int(best_row["epochs"]),
        "batch_size": int(best_row["batch_size"])
    }

    best_threshold = best_row["threshold"]
    print(f"Sensor: {sensor_type}")
    print("Parameters:")
    print(best_params)

    print(f"Threshold : {best_threshold:.6f}")
    print(f"Accuracy  : {best_row['Accuracy']:.4f}")
    print(f"Precision : {best_row['Precision']:.4f}")
    print(f"Recall    : {best_row['Recall']:.4f}")
    print(f"ROC AUC   : {best_row['ROC AUC']:.4f}")
    print(f"F1        : {best_row['F1']:.4f}")

    print("All configurations:")
    print(results_df.sort_values("F1", ascending=False).head(10))

    return best_params, best_threshold, results_df


In [27]:

param_grid = {
    "encoding_dim": [8, 16, 32],
    "epochs": [30, 50],
    "batch_size": [32, 64]
}

In [28]:
best_params_acc, threshold_acc, tuning_acc = tune_autoencoder(
    X_train_source_acc,
    X_val_acc,
    X_train_source,
    X_val,
    "ACC source",
    param_grid
)

Sensor: ACC source
Parameters:
{'encoding_dim': 32, 'epochs': 50, 'batch_size': 32}
Threshold : 0.003053
Accuracy  : 0.8405
Precision : 0.7883
Recall    : 0.9310
ROC AUC   : 0.8749
F1        : 0.8538
All configurations:
        sensor  batch_size  encoding_dim  epochs  threshold  Accuracy  \
5   ACC source          32            32      50   0.003053  0.840517   
4   ACC source          32            32      30   0.001702  0.827586   
1   ACC source          32             8      50   0.051245  0.827586   
7   ACC source          64             8      50   0.053918  0.814655   
10  ACC source          64            32      30   0.002862  0.814655   
9   ACC source          64            16      50   0.004248  0.831897   
3   ACC source          32            16      50   0.005380  0.780172   
11  ACC source          64            32      50   0.001373  0.784483   
0   ACC source          32             8      30   0.052166  0.762931   
8   ACC source          64            16      30  

In [29]:
best_params_mic, threshold_mic, tuning_mic = tune_autoencoder(
    X_train_source_mic,
    X_val_mic,
    X_train_source,
    X_val,
    "MIC source",
    param_grid
)

Sensor: MIC source
Parameters:
{'encoding_dim': 8, 'epochs': 30, 'batch_size': 64}
Threshold : 0.001753
Accuracy  : 0.5388
Precision : 0.5205
Recall    : 0.9828
ROC AUC   : 0.5697
F1        : 0.6806
All configurations:
        sensor  batch_size  encoding_dim  epochs  threshold  Accuracy  \
6   MIC source          64             8      30   0.001753  0.538793   
10  MIC source          64            32      30   0.000493  0.534483   
0   MIC source          32             8      30   0.000321  0.530172   
11  MIC source          64            32      50   0.000389  0.521552   
1   MIC source          32             8      50   0.000267  0.517241   
9   MIC source          64            16      50   0.000097  0.517241   
5   MIC source          32            32      50   0.003280  0.594828   
4   MIC source          32            32      30   0.000374  0.521552   
3   MIC source          32            16      50   0.000527  0.517241   
7   MIC source          64             8      50   

In [30]:
best_params_gyro, threshold_gyro, tuning_gyro = tune_autoencoder(
    X_train_source_gyro,
    X_val_gyro,
    X_train_source,
    X_val,
    "GYRO source",
    param_grid
)

Sensor: GYRO source
Parameters:
{'encoding_dim': 8, 'epochs': 30, 'batch_size': 64}
Threshold : 0.085766
Accuracy  : 0.7716
Precision : 0.7032
Recall    : 0.9397
ROC AUC   : 0.8315
F1        : 0.8044
All configurations:
         sensor  batch_size  encoding_dim  epochs  threshold  Accuracy  \
6   GYRO source          64             8      30   0.085766  0.771552   
0   GYRO source          32             8      30   0.089964  0.741379   
1   GYRO source          32             8      50   0.082297  0.724138   
7   GYRO source          64             8      50   0.066777  0.702586   
9   GYRO source          64            16      50   0.011143  0.668103   
11  GYRO source          64            32      50   0.001906  0.659483   
3   GYRO source          32            16      50   0.016929  0.620690   
8   GYRO source          64            16      30   0.005371  0.616379   
10  GYRO source          64            32      30   0.001674  0.577586   
2   GYRO source          32            1

In [31]:
def run_autoencoder(X_train, X_test, test_df, sensor_type, best_params, threshold, X_train_target=None):
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = build_autoencoder(
        input_dim=X_train_scaled.shape[1],
        encoding_dim=best_params["encoding_dim"]
    )

    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    model.fit(
        X_train_scaled, X_train_scaled,
        epochs=best_params["epochs"],
        batch_size=best_params["batch_size"],
        validation_split=0.1,
        callbacks=[early_stop],
        verbose=0
    )

    if X_train_target is not None and not X_train_target.empty:
        X_tgt_scaled = scaler.transform(X_train_target)
        tgt_pred = model.predict(X_tgt_scaled, verbose=0)
        tgt_mse = np.mean(np.power(X_tgt_scaled - tgt_pred, 2), axis=1)
        used_threshold = np.percentile(tgt_mse, 95)
    else:
        used_threshold = threshold

    X_test_pred = model.predict(X_test_scaled, verbose=0)
    test_scores = np.mean(np.power(X_test_scaled - X_test_pred, 2), axis=1)

    results = pd.DataFrame({"segment_id": test_df["segment_id"].values,
                            "anomaly_label": test_df["anomaly_label"].values,
                            "score": test_scores})

    segment_scores = (results.groupby("segment_id").agg(score=("score", "max"),
                                                        anomaly_label=("anomaly_label", "first")).reset_index())
    segment_scores["y_true"] = (segment_scores["anomaly_label"] == "loosescrewsA").astype(int)
    segment_scores["y_pred"] = (segment_scores["score"] >= used_threshold).astype(int)

    y_true = segment_scores["y_true"]
    y_pred = segment_scores["y_pred"]

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    roc_auc = roc_auc_score(y_true, segment_scores["score"])

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print(f"{sensor_type}")
    print(f"Threshold: {used_threshold:.4f}")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC AUC  : {roc_auc:.4f}")
    print(cm)

    metrics = {"Sensors": sensor_type,
               "Accuracy": accuracy,
               "Precision": precision,
               "Recall": recall,
               "F1": f1,
               "ROC_AUC": roc_auc,
               "TN": cm[0, 0],
               "FP": cm[0, 1],
               "FN": cm[1, 0],
               "TP": cm[1, 1]}

    return metrics, segment_scores, model


Trenowanie
i
testowanie
osobnych
modeli(Autoenkodery)
wyłącznie
na
domenie
source

In [32]:

metrics_acc_source, pred_acc_source, model_acc_source = run_autoencoder(
    X_train_source_acc, X_val_acc, X_val, "ACC - source", best_params_acc, threshold_acc
)
metrics_mic_source, pred_mic_source, model_mic_source = run_autoencoder(
    X_train_source_mic, X_val_mic, X_val, "MIC - source", best_params_mic, threshold_mic
)
metrics_gyro_source, pred_gyro_source, model_gyro_source = run_autoencoder(
    X_train_source_gyro, X_val_gyro, X_val, "GYRO - source", best_params_gyro, threshold_gyro
)

ACC - source
Threshold: 0.0031
Accuracy : 0.7974
Precision: 0.7379
Recall   : 0.9224
F1-score : 0.8199
ROC AUC  : 0.8992
[[ 78  38]
 [  9 107]]
MIC - source
Threshold: 0.0018
Accuracy : 0.5560
Precision: 0.5442
Recall   : 0.6897
F1-score : 0.6084
ROC AUC  : 0.5552
[[49 67]
 [36 80]]
GYRO - source
Threshold: 0.0858
Accuracy : 0.6724
Precision: 0.6075
Recall   : 0.9741
F1-score : 0.7483
ROC AUC  : 0.8107
[[ 43  73]
 [  3 113]]


Segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
jeden
z
trzech
modeli
wykryje
anomalię(Source -> Source)

In [33]:
final_source_results = (pred_acc_source[["segment_id", "y_true", "y_pred"]].rename(columns={"y_pred": "pred_acc"})
                        .merge(pred_mic_source[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_mic"}),
                               on="segment_id")
                        .merge(pred_gyro_source[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_gyro"}),
                               on="segment_id"))

final_source_results["y_pred_final"] = (
        final_source_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 1).astype(int)
y_true_source = final_source_results["y_true"]
y_pred_source_final = final_source_results["y_pred_final"]

print("Final Source Results (at least 1 sensor):")
print(f"Accuracy : {accuracy_score(y_true_source, y_pred_source_final):.4f}")
print(f"Precision: {precision_score(y_true_source, y_pred_source_final, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_true_source, y_pred_source_final, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_true_source, y_pred_source_final, zero_division=0):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_true_source, y_pred_source_final, labels=[0, 1]))

Final Source Results (at least 1 sensor):
Accuracy : 0.5690
Precision: 0.5370
Recall   : 1.0000
F1-score : 0.6988
Confusion matrix:
 [[ 16 100]
 [  0 116]]


Segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
dwa
z
trzech
modeli
wykryją
anomalię(Source -> Source)

In [34]:
final_source_results["y_pred_final"] = (
        final_source_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 2).astype(int)
y_pred_source_final = final_source_results["y_pred_final"]

print("Final Source Results (majority voting):")
print(f"Accuracy : {accuracy_score(y_true_source, y_pred_source_final):.4f}")
print(f"Precision: {precision_score(y_true_source, y_pred_source_final, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_true_source, y_pred_source_final, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_true_source, y_pred_source_final, zero_division=0):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_true_source, y_pred_source_final, labels=[0, 1]))

Final Source Results (majority voting):
Accuracy : 0.7284
Precision: 0.6514
Recall   : 0.9828
F1-score : 0.7835
Confusion matrix:
 [[ 55  61]
 [  2 114]]


Trenowanie
osobnych
modeli
dla
każdego
czujnika
wyłącznie
z
wykorzystaniem
domeny
source, a
testowanie
na
target

In [42]:
metrics_acc_tgt, pred_acc_tgt, model_acc_tgt = run_autoencoder(
    X_train_source_acc, X_test_acc, X_test, "ACC source -> target", best_params_acc, threshold_acc,X_train_target=X_train_target_acc
)
metrics_mic_tgt, pred_mic_tgt, model_mic_tgt = run_autoencoder(
    X_train_source_mic, X_test_mic, X_test, "MIC source -> target", best_params_mic, threshold_mic,
    X_train_target=X_train_target_mic
)
metrics_gyro_tgt, pred_gyro_tgt, model_gyro_tgt = run_autoencoder(
    X_train_source_gyro, X_test_gyro, X_test, "GYRO source -> target", best_params_gyro, threshold_gyro,
    X_train_target=X_train_target_gyro
)


ACC source -> target
Threshold: 0.0072
Accuracy : 0.5991
Precision: 0.5732
Recall   : 0.7759
F1-score : 0.6593
ROC AUC  : 0.6131
[[49 67]
 [26 90]]
MIC source -> target
Threshold: 0.0024
Accuracy : 0.5043
Precision: 0.5026
Recall   : 0.8276
F1-score : 0.6254
ROC AUC  : 0.4957
[[21 95]
 [20 96]]
GYRO source -> target
Threshold: 0.1362
Accuracy : 0.5302
Precision: 0.5196
Recall   : 0.8017
F1-score : 0.6305
ROC AUC  : 0.5126
[[30 86]
 [23 93]]


Segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
jeden
z
trzech
modeli
wykryje
anomalię(Source -> Target)

In [43]:
final_tgt_results = (pred_acc_tgt[["segment_id", "y_true", "y_pred"]].rename(columns={"y_pred": "pred_acc"})
                     .merge(pred_mic_tgt[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_mic"}),
                            on="segment_id")
                     .merge(pred_gyro_tgt[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_gyro"}),
                            on="segment_id"))

final_tgt_results["y_pred_final"] = (final_tgt_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 1).astype(
    int)
y_true_tgt = final_tgt_results["y_true"]
y_pred_tgt_final = final_tgt_results["y_pred_final"]

print("Final Results source -> target (at least 1 sensor):")
print(f"Accuracy : {accuracy_score(y_true_tgt, y_pred_tgt_final):.4f}")
print(f"Precision: {precision_score(y_true_tgt, y_pred_tgt_final, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_true_tgt, y_pred_tgt_final, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_true_tgt, y_pred_tgt_final, zero_division=0):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_true_tgt, y_pred_tgt_final, labels=[0, 1]))


Final Results source -> target (at least 1 sensor):
Accuracy : 0.5172
Precision: 0.5088
Recall   : 1.0000
F1-score : 0.6744
Confusion matrix:
 [[  4 112]
 [  0 116]]


Teraz
segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
dwa
z
trzech
modeli
wykryją
anomalię(Source -> Target)

In [44]:

final_tgt_results["y_pred_final"] = (final_tgt_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 2).astype(
    int)
y_pred_tgt_final = final_tgt_results["y_pred_final"]

print("Final Results source -> target (majority voting):")
print(f"Accuracy : {accuracy_score(y_true_tgt, y_pred_tgt_final):.4f}")
print(f"Precision: {precision_score(y_true_tgt, y_pred_tgt_final, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_true_tgt, y_pred_tgt_final, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_true_tgt, y_pred_tgt_final, zero_division=0):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_true_tgt, y_pred_tgt_final, labels=[0, 1]))


Final Results source -> target (majority voting):
Accuracy : 0.5647
Precision: 0.5381
Recall   : 0.9138
F1-score : 0.6773
Confusion matrix:
 [[ 25  91]
 [ 10 106]]


Trenowanie
i
testowanie
osobnych
modeli
dla
każdego
czujnika
z
wykorzystaniem
całych
zbiorów
treningowego
i
testowego

In [45]:
metrics_acc_all, pred_acc_all, model_acc_all = run_autoencoder(
    X_train_acc, X_test_full_acc, X_test_df, "ACC (ALL)",
    best_params_acc, threshold_acc,
    X_train_target=X_train_acc
)
metrics_mic_all, pred_mic_all, model_mic_all = run_autoencoder(
    X_train_mic, X_test_full_mic, X_test_df, "MIC (ALL)",
    best_params_mic, threshold_mic,
    X_train_target=X_train_mic
)
metrics_gyro_all, pred_gyro_all, model_gyro_all = run_autoencoder(
    X_train_gyro, X_test_full_gyro, X_test_df, "GYRO (ALL)",
    best_params_gyro, threshold_gyro,
    X_train_target=X_train_gyro
)

ACC (ALL)
Threshold: 0.0019
Accuracy : 0.5086
Precision: 0.5043
Recall   : 1.0000
F1-score : 0.6705
ROC AUC  : 0.6602
[[  4 228]
 [  0 232]]
MIC (ALL)
Threshold: 0.0036
Accuracy : 0.5151
Precision: 0.5104
Recall   : 0.7414
F1-score : 0.6046
ROC AUC  : 0.5341
[[ 67 165]
 [ 60 172]]
GYRO (ALL)
Threshold: 0.0564
Accuracy : 0.5797
Precision: 0.5433
Recall   : 1.0000
F1-score : 0.7041
ROC AUC  : 0.5907
[[ 37 195]
 [  0 232]]


Segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
jeden
z
trzech
modeli
wykryje
anomalię(ALL -> ALL)

In [39]:
final_all_results = (pred_acc_all[["segment_id", "y_true", "y_pred"]].rename(columns={"y_pred": "pred_acc"})
                     .merge(pred_mic_all[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_mic"}),
                            on="segment_id")
                     .merge(pred_gyro_all[["segment_id", "y_pred"]].rename(columns={"y_pred": "pred_gyro"}),
                            on="segment_id"))

final_all_results["y_pred_final"] = (final_all_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 1).astype(
    int)
y_true_all = final_all_results["y_true"]
y_pred_all_final = final_all_results["y_pred_final"]

print("Final Results ALL -> ALL (at least 1 sensor):")
print(f"Accuracy : {accuracy_score(y_true_all, y_pred_all_final):.4f}")
print(f"Precision: {precision_score(y_true_all, y_pred_all_final, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_true_all, y_pred_all_final, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_true_all, y_pred_all_final, zero_division=0):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_true_all, y_pred_all_final, labels=[0, 1]))

Final Results ALL -> ALL (at least 1 sensor):
Accuracy : 0.5000
Precision: 0.5000
Recall   : 1.0000
F1-score : 0.6667
Confusion matrix:
 [[  0 232]
 [  0 232]]


Teraz
segment
jest
klasyfikowany
jako
anomalia, jeśli
co
najmniej
dwa
z
trzech
modeli
wykryją
anomalię(ALL -> ALL)

In [40]:


final_all_results["y_pred_final"] = (final_all_results[["pred_acc", "pred_mic", "pred_gyro"]].sum(axis=1) >= 2).astype(
    int)
y_pred_all_final = final_all_results["y_pred_final"]

print("Final Results ALL -> ALL (majority voting):")
print(f"Accuracy : {accuracy_score(y_true_all, y_pred_all_final):.4f}")
print(f"Precision: {precision_score(y_true_all, y_pred_all_final, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_true_all, y_pred_all_final, zero_division=0):.4f}")
print(f"F1-score : {f1_score(y_true_all, y_pred_all_final, zero_division=0):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_true_all, y_pred_all_final, labels=[0, 1]))

Final Results ALL -> ALL (majority voting):
Accuracy : 0.5345
Precision: 0.5183
Recall   : 0.9741
F1-score : 0.6766
Confusion matrix:
 [[ 22 210]
 [  6 226]]
